# Week 1: Foundations — Cleaning & Exploring the Applicant Dataset

**Internship:** Data Science Internship — NextGen Learners

**Objective:**
To explore, clean, and validate an applicant dataset using Python and Pandas.

**Tasks:**

* Explore the dataset and identify data quality issues.
* Handle missing values and duplicate records.
* Standardize text fields.
* Correct data types and date formats.
* Export the cleaned dataset and summarize the results.



#1 Get Data

In [1]:
import pandas as pd

In [12]:
from google.colab import files
uploaded = files.upload()

Saving applicants.csv to applicants (1).csv


In [36]:
df = pd.read_csv('/content/applicants.csv')
print("Dataset loaded successfully!")

Dataset loaded successfully!


In [37]:
df.head(20)

,Applicant Name,Email,Phone,Domain Applied,University,Application Date,Status
0,Karim Reda,karim.reda41@example.com,01038270233,data analytics,Ain Shams University,2026-09-07,SELECTED
1,Salma Hassan,salma.hassan23@example.com,01088961459,cyber security,Helwan University,2026-06-26,Under Review
2,Aya Saleh,aya.saleh56@example.com,01092613013,CYBERSECURITY,Menoufia University,18/09/2026,UNDER REVIEW
3,Hana Sherif,hana.sherif73@example.com,0105 3868501,Data Analytics,Ain Shams University,2026-08-16,Selected
4,Khaled Khalil,khaled.khalil1@example.com,0101-3356886,data analytics,Mansoura University,2026-06-05,Rejected
5,Malak Sherif,malak.sherif27@example.com,01017270733,Data Science,Menoufia University,2026-07-26,under review
6,Reem Khalil,reem.khalil40@example.com,01028024248,Data Analytics,Cairo University,19/09/2026,UNDER REVIEW
7,Malak Mohamed,malak.mohamed68@example.com,01037760841,UI/UX,Helwan University,06/05/2026,Rejected
8,Mahmoud Mohamed,MAHMOUD.MOHAMED11@EXAMPLE.COM,01052339391,UI UX,Cairo University,2026-06-24,REJECTED
9,Youssef Mohamed,youssef.mohamed45@example.com,01051373735,Data Analytics,Zagazig University,2026-06-23,rejected


#2 First Look — Understand Before You Clean

In [38]:
df.shape

(94, 7)

In [39]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 94 entries, 0 to 93
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   Applicant Name    94 non-null     object
 1   Email             91 non-null     object
 2   Phone             94 non-null     object
 3   Domain Applied    94 non-null     object
 4   University        90 non-null     object
 5   Application Date  94 non-null     object
 6   Status            91 non-null     object
dtypes: object(7)
memory usage: 5.3+ KB


In [40]:
df.describe(include='all')

,Applicant Name,Email,Phone,Domain Applied,University,Application Date,Status
count,94,91,94,94,90,94,91
unique,89,87,90,25,10,79,12
top,Hana Said,hana.said5@example.com,01023718431,Web Development,Cairo University,11/07/2026,UNDER REVIEW
freq,2,2,2,8,16,3,13


In [41]:
df.isnull().sum()

,0
Applicant Name,0
Email,3
Phone,0
Domain Applied,0
University,4
Application Date,0
Status,3


In [42]:
df.duplicated().sum()

np.int64(3)

The dataset contains **94 rows and 7 columns**. Email has 3 missing values, University has 4, and Status has 3. All columns are stored as `object`, including Application Date, which should be converted to `datetime`. Duplicate records need to be checked.


#3 Handle Missing Values

In [43]:
# Check missing values before cleaning
print("Missing values before cleaning:")
print(df.isnull().sum())

Missing values before cleaning:
Applicant Name      0
Email               3
Phone               0
Domain Applied      0
University          4
Application Date    0
Status              3
dtype: int64


In [44]:
# Handle missing Email values
df['Email'] = df['Email'].fillna('not_provided@example.com')

# Handle missing University values
df['University'] = df['University'].fillna('Not Specified')

# Handle missing Status values
df['Status'] = df['Status'].fillna('Under Review')

In [45]:
# Check missing values after cleaning
print("\nMissing values after cleaning:")
print(df.isnull().sum())


Missing values after cleaning:
Applicant Name      0
Email               0
Phone               0
Domain Applied      0
University          0
Application Date    0
Status              0
dtype: int64


* **Applicant Name:** No missing values; no action needed.
* **Email:** Filled 3 missing values with `not_provided@example.com` to preserve rows.
* **University:** Filled 4 missing values with `Not Specified` because this information is not essential.
* **Status:** Filled 3 missing values with `Under Review`, assuming unrecorded applications are still being processed.


# 4 Remove Duplicate Applicants

In [46]:
# Check exact duplicates
print("Exact duplicates:", df.duplicated().sum())

# Remove exact duplicates
df = df.drop_duplicates()

# Check duplicate emails
print("Duplicate emails:", df['Email'].duplicated().sum())

# Remove duplicate applications by email
df = df.drop_duplicates(subset=['Email'], keep='first')

print("Dataset shape after removing duplicates:", df.shape)

Exact duplicates: 3
Duplicate emails: 3
Dataset shape after removing duplicates: (88, 7)


I removed exact duplicate rows and checked for repeated applications using Email. Email is a more reliable identifier than Name because different applicants may share the same name. I kept the first application for each email to avoid duplicate records.


#5 Standardize Text Fields

In [47]:
# Strip whitespace from text columns
for col in df.select_dtypes(include='object').columns:
    df[col] = df[col].str.strip()

# Standardize names
df['Applicant Name'] = df['Applicant Name'].str.title()

# Standardize domain names for mapping
df['Domain Applied'] = df['Domain Applied'].str.lower()

# Map domain variations (update based on actual unique values)
domain_mapping = {
    'web dev': 'Web Development',
    'webdev': 'Web Development',
    'web development': 'Web Development',
}

df['Domain Applied'] = df['Domain Applied'].replace(domain_mapping)

# Standardize status
df['Status'] = df['Status'].str.title()

# Check actual domain values
print(df['Domain Applied'].unique())
print(df['Status'].unique())

['data analytics' 'cyber security' 'cybersecurity' 'data science' 'ui/ux'
 'ui ux' 'datascience' 'Web Development' 'data-analytics' 'ui/ux design']
['Selected' 'Under Review' 'Rejected']


I removed extra whitespace, standardized text casing, and mapped inconsistent domain names to consistent labels. I also standardized application statuses to ensure accurate grouping and analysis.


#6 Fix Data Types

In [48]:
# Convert Application Date to datetime
df['Application Date'] = pd.to_datetime(df['Application Date'], errors='coerce')

# Clean phone numbers
df['Phone'] = (df['Phone'].str.replace('-', '', regex=False).str.replace(' ', '', regex=False))

# Check missing dates
print("Missing dates:", df['Application Date'].isnull().sum())

# Verify data types
df.info()

Missing dates: 47
<class 'pandas.core.frame.DataFrame'>
Index: 88 entries, 0 to 93
Data columns (total 7 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   Applicant Name    88 non-null     object        
 1   Email             88 non-null     object        
 2   Phone             88 non-null     object        
 3   Domain Applied    88 non-null     object        
 4   University        88 non-null     object        
 5   Application Date  41 non-null     datetime64[ns]
 6   Status            88 non-null     object        
dtypes: datetime64[ns](1), object(6)
memory usage: 5.5+ KB


I converted `Application Date` to `datetime` for accurate date operations and removed spaces and dashes from `Phone` numbers while keeping them as strings to preserve leading zeros.


#7 Data Quality Summary

**Starting Dataset:** 94 rows, 7 columns.

**Issues Found:**

* Missing values: Email (3), University (4), Status (3).
* Potential duplicate rows and applications.
* Inconsistent text formatting in domain names and statuses.
* Application Date stored as text instead of datetime.
* Inconsistent phone number formatting.

**Actions Taken:**

* Filled missing Email with `not_provided@example.com`, University with `Not Specified`, and Status with `Under Review`.
* Removed exact duplicate rows and duplicate applications identified by Email.
* Standardized domain names and application statuses.
* Converted Application Date to datetime.
* Removed spaces and dashes from phone numbers.

**Final Dataset:** The dataset contains 7 columns. The final row count depends on the number of duplicates removed. Data types and text formatting were standardized, and missing values were handled according to the chosen rules.




# 8 Export Cleaned Dataset

In [49]:
df.to_csv('applicants_cleaned.csv', index=False)

print("Cleaned dataset exported successfully!")

Cleaned dataset exported successfully!


In [50]:
import os

print(os.path.exists('applicants_cleaned.csv'))

True


## Final Submission Checklist

* [ ] Notebook runs from beginning to end without errors.
* [ ] Missing value handling decisions are documented.
* [ ] Missing values and duplicate records are checked.
* [ ] Domain Applied contains exactly 5 standardized values.
* [ ] Application Date has the correct datetime type.
* [ ] Phone numbers have consistent formatting.
* [ ] Data Quality Summary is included.
* [ ] `applicants_cleaned.csv` is exported and added to the GitHub repository.
* [ ] All cleaning operations are performed using code, not manual Excel edits.
